<a href="https://colab.research.google.com/github/Przemo75/git_training/blob/master/HAKI_sortowanie_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

# === 1. Wczytanie plików ===
new_file = '/content/_sort_new.csv'
old_file = '/content/_sort_old.csv'

df_new = pd.read_csv(new_file)
df_old = pd.read_csv(old_file)

columns = ['MARKA', 'MODEL', 'ROCZNIK', 'RODZAJ NADWOZIA']

# === 2. Tworzenie maski zmian ===
diff_mask = pd.DataFrame(False, index=df_new.index, columns=columns)

for col in columns:
    old_values = set(df_old[col].dropna().unique())
    diff_mask[col] = ~df_new[col].isin(old_values)

print("Nowe wartości:", diff_mask.sum().sum())

Nowe wartości: 2364


In [13]:
output_file = '/content/nowe_wartosci.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_new.to_excel(writer, index=False, sheet_name='Wynik')

    workbook = writer.book
    worksheet = writer.sheets['Wynik']

    from openpyxl.styles import PatternFill
    red_fill = PatternFill(start_color='FF0000', end_color='FF0000', fill_type='solid')

    for row in range(len(df_new)):
        for col in range(len(df_new.columns)):
            if diff_mask.iat[row, col]:
                worksheet.cell(row=row+2, column=col+1).fill = red_fill

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
rows = []

for col in columns:
    new_vals = df_new.loc[diff_mask[col], col].dropna().unique()

    for val in new_vals:
        rows.append({
            "kolumna": col,
            "nowa_wartość": val
        })

df_new_values = pd.DataFrame(rows)

In [15]:
summary = {
    "nowe_marki": df_new.loc[diff_mask["MARKA"], "MARKA"].nunique(),
    "nowe_modele": df_new.loc[diff_mask["MODEL"], "MODEL"].nunique(),
    "nowe_roczniki": df_new.loc[diff_mask["ROCZNIK"], "ROCZNIK"].nunique(),
    "nowe_nadwozia": df_new.loc[diff_mask["RODZAJ NADWOZIA"], "RODZAJ NADWOZIA"].nunique(),
}

df_summary = pd.DataFrame(list(summary.items()), columns=["kategoria", "liczba"])

In [16]:
with pd.ExcelWriter("porownanie.xlsx", engine="openpyxl") as writer:
    df_new.to_excel(writer, sheet_name="NEW_DATA", index=False)
    df_new_values.to_excel(writer, sheet_name="NOWE_WARTOSCI", index=False)
    df_summary.to_excel(writer, sheet_name="PODSUMOWANIE", index=False)